# S7 · Ingesta de telemetría real con OpenSky y Kafka

**Integrante** · Leonardo Jesus Huaman Arhuata

**Dimensión** · volumen de aeronaves en vuelo

Equipo 04 · AeroStreamPeru · Ciclo 9

El productor compartido `productor_opensky.py`, en esta misma carpeta, consulta capturas de OpenSky REST autenticadas y las publica en Kafka. Esta práctica verifica la salida y aplica controles al dato recibido.

## 1. Consumir telemetría y metadatos de captura

`opensky.telemetria.v1` lleva un estado por mensaje y usa `icao24` como clave. `opensky.capturas.v1` registra si la captura llegó y cuántos estados se publicaron. El consumidor lee una cola acotada de cada partición sin guardar offsets de grupo.

In [1]:
from collections import Counter
from datetime import datetime, timezone
import json
import math
import re
from time import monotonic

from kafka import KafkaConsumer, TopicPartition

KAFKA = "kafka:9092"
TOPIC_CAPTURAS = "opensky.capturas.v1"
TOPIC_TELEMETRIA = "opensky.telemetria.v1"


def leer_cola(topico, maximo_por_particion):
    consumidor = KafkaConsumer(
        bootstrap_servers=KAFKA,
        enable_auto_commit=False,
        max_poll_records=500,
        key_deserializer=lambda dato: dato.decode("utf-8"),
        value_deserializer=lambda dato: json.loads(dato),
    )
    try:
        particiones = consumidor.partitions_for_topic(topico)
        if particiones is None:
            raise RuntimeError(f"No se encontró {topico}. Ejecuta el productor de OpenSky.")
        destinos = [TopicPartition(topico, indice) for indice in sorted(particiones)]
        consumidor.assign(destinos)
        finales = consumidor.end_offsets(destinos)
        principios = consumidor.beginning_offsets(destinos)
        for destino in destinos:
            consumidor.seek(destino, max(principios[destino], finales[destino] - maximo_por_particion))

        registros = []
        limite = monotonic() + 15
        while any(consumidor.position(destino) < finales[destino] for destino in destinos):
            if monotonic() >= limite:
                raise TimeoutError(f"No se pudo leer la cola completa de {topico}")
            for destino, lote in consumidor.poll(timeout_ms=500).items():
                registros.extend(mensaje for mensaje in lote if mensaje.offset < finales[destino])
        return registros
    finally:
        consumidor.close()

In [2]:
capturas = [mensaje.value for mensaje in leer_cola(TOPIC_CAPTURAS, 30)]
for captura in sorted(capturas, key=lambda dato: dato["time"])[-5:]:
    print(captura["capture_id"], captura["estado"], captura["estados_publicados"])

observadas = [captura for captura in capturas if captura["estado"] in {"ok", "parcial"}]
if not observadas:
    raise RuntimeError("Kafka aún no tiene una captura exitosa de OpenSky")
captura = max(observadas, key=lambda dato: dato["time"])
instantes = sorted({dato["time"] for dato in observadas})
if len(instantes) >= 2:
    print(f"Separación de las dos últimas capturas: {instantes[-1] - instantes[-2]} segundos")
assert captura["time"] % captura["cadencia_segundos"] == 0

mensajes = [
    mensaje for mensaje in leer_cola(TOPIC_TELEMETRIA, 300)
    if mensaje.value["capture_id"] == captura["capture_id"]
]
assert len(mensajes) >= captura["estados_publicados"], "La cola consultada no cubre toda la captura"
print(f"Captura UTC: {datetime.fromtimestamp(captura['time'], timezone.utc).isoformat()}")
print(f"Cadencia configurada: {captura['cadencia_segundos']} segundos")
print(f"Distribución Kafka por partición: {dict(sorted(Counter(m.partition for m in mensajes).items()))}")

for mensaje in mensajes:
    evento = mensaje.value
    assert evento["version_esquema"] == 1
    assert re.fullmatch(r"[0-9a-f]{6}", evento["icao24"])
    assert mensaje.key == evento["icao24"]
    assert evento["event_id"] == f"{captura['time']}:{evento['icao24']}"
    assert evento["time"] == captura["time"]
    assert isinstance(evento["onground"], bool)

estados = {mensaje.value["event_id"]: mensaje.value for mensaje in mensajes}
print(f"Estados únicos recibidos: {len(estados)}")

/tmp/ipykernel_275/3669172162.py:16: DeprecationWarning: key_deserializer does not implement kafka.serializer.Deserializer
  consumidor = KafkaConsumer(
/tmp/ipykernel_275/3669172162.py:16: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumidor = KafkaConsumer(


opensky:1795540080 ok 6
opensky:1795540200 ok 8
Separación de las dos últimas capturas: 120 segundos
Captura UTC: 2026-11-24T17:10:00+00:00
Cadencia configurada: 120 segundos
Distribución Kafka por partición: {0: 1, 1: 2, 2: 5}
Estados únicos recibidos: 8


## 2. Controles de mi dimensión

Se valida la estructura del mensaje por separado de su rango. La prueba de alerta altera una copia en memoria de una lectura real. La copia de prueba se queda en el notebook.

In [3]:
def posicion_valida(estado):
    latitud, longitud = estado["lat"], estado["lon"]
    return (
        latitud is not None and longitud is not None
        and -18.35 <= latitud <= -0.04
        and -81.33 <= longitud <= -68.65
    )


assert all(posicion_valida(estado) for estado in estados.values())
en_vuelo = {estado["icao24"] for estado in estados.values() if not estado["onground"]}
print(f"Aeronaves distintas en vuelo en esta captura: {len(en_vuelo)}")

if estados:
    lectura_real = next(iter(estados.values()))
    copia_de_prueba = {**lectura_real, "lat": -200.0}
    assert posicion_valida(lectura_real) and not posicion_valida(copia_de_prueba)
    print("Alerta de rango verificada con una copia local: latitud fuera del área.")

Aeronaves distintas en vuelo en esta captura: 8
Alerta de rango verificada con una copia local: latitud fuera del área.


## 3. Interpretación y evidencia

Una captura con cero aeronaves válidas puede producir un conteo de cero. Una solicitud fallida se registra aparte y representa cobertura faltante. La clave ICAO24 permite comprobar la distribución y seguir estados de la misma aeronave.

Para la evidencia, registra los offsets y la distribución de particiones en Kafka UI y la salida de la prueba de rango.